# ShortsLM — Personal YouTube Shorts & Instagram Reels Feature Extraction

**Clean, resumable Google Colab notebook**

This notebook reads your personal URL lists from Google Drive, downloads one video at a time using the repository's supported download libraries, runs the existing ShortsLM analyzer, deletes temporary media, and pushes one feature JSON per video to a **private GitHub repository**.

### Important platform-access facts
- YouTube may require an authenticated browser cookie file when Colab requests are challenged. The notebook does not try to bypass YouTube's bot checks or fabricate player clients.
- Instagram public Reels may download without login. If Instagram requires authentication, the notebook validates a genuine Instaloader session file and can help you refresh it through Instaloader's normal login flow.
- Authentication is not guaranteed for every URL. Private, deleted, age-restricted, geo-restricted, or otherwise inaccessible content will remain in `remaining.txt`.
- Use this only for videos you own or are authorized to download and analyze.

**Security:** feature JSON may contain transcripts and OCR text. The notebook refuses to push to a public GitHub repository. Never commit session files, cookie files, passwords, or tokens.
\n**Token safety:** the previous uploaded notebook exposed a GitHub token in its code. Revoke that token before using this replacement notebook.\n

## 1. Mount Drive and create persistent workspace

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, re, json, time, shutil, tempfile, subprocess, base64, hashlib
from datetime import datetime, timezone
from urllib.parse import quote
import requests

DRIVE_ROOT = Path("/content/drive/MyDrive/ShortsLM_Personal_Pilot")
INPUT_DIR = DRIVE_ROOT / "input"
STATE_DIR = DRIVE_ROOT / "state"
FEATURES_DIR = DRIVE_ROOT / "features"
SECRETS_DIR = DRIVE_ROOT / "secrets"

for folder in (INPUT_DIR, STATE_DIR, FEATURES_DIR, SECRETS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

COMPLETED_FILE = STATE_DIR / "completed.txt"
REMAINING_FILE = STATE_DIR / "remaining.txt"
FAILED_LOG = STATE_DIR / "failed.jsonl"
PUSH_LOG = STATE_DIR / "github_push_log.jsonl"

print("Persistent workspace:", DRIVE_ROOT)
print("Input folder:", INPUT_DIR)
print("Feature folder:", FEATURES_DIR)


## 2. Put your URL TXT files in Drive

In [ ]:
from google.colab import files

def find_input_file(candidates):
    wanted = {name.casefold() for name in candidates}
    for path in INPUT_DIR.iterdir():
        if path.is_file() and path.name.casefold() in wanted:
            return path
    return None

ig_file = find_input_file([
    "Instagram_reels.txt", "instagram_reels.txt", "instagram reels.txt"
])
yt_file = find_input_file([
    "YouTube_shorts.txt", "youtube_shorts.txt",
    "YouTube_shorts_YouTube_shorts.txt", "youtube_shorts_youtube_shorts.txt"
])

missing = []
if ig_file is None:
    missing.append("Instagram_reels.txt")
if yt_file is None:
    missing.append("YouTube_shorts.txt")

if missing:
    print("Upload the missing TXT file(s):", ", ".join(missing))
    uploaded = files.upload()
    for filename, data in uploaded.items():
        (INPUT_DIR / filename).write_bytes(data)

ig_file = find_input_file([
    "Instagram_reels.txt", "instagram_reels.txt", "instagram reels.txt"
])
yt_file = find_input_file([
    "YouTube_shorts.txt", "youtube_shorts.txt",
    "YouTube_shorts_YouTube_shorts.txt", "youtube_shorts_youtube_shorts.txt"
])

if ig_file is None and yt_file is None:
    raise FileNotFoundError(
        f"No URL TXT files found in {INPUT_DIR}. Upload at least one file."
    )

print("Instagram URL file:", ig_file)
print("YouTube URL file:", yt_file)


## 3. Clone ShortsLM and install its real dependencies

In [ ]:
REPO_URL = "https://github.com/Arhanpg/Shorts-Language-Model.git"
PROJECT_DIR = Path("/content/Shorts-Language-Model")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
    check=True
)

# Install the repository's declared extras. Do not patch downloader.py at runtime.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e",
     ".[audio,ocr,discovery,legacy-discovery,instagram-fallback]"],
    cwd=PROJECT_DIR,
    check=True
)

# Ensure yt-dlp is current and use its supported default dependencies.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "yt-dlp[default]"],
    check=True
)

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "nodejs"], check=True)

print("Repository installed:", PROJECT_DIR)
print("Node:", shutil.which("node"))
print("FFmpeg:", shutil.which("ffmpeg"))


## 4. Load ShortsLM analyzer and configure feature extraction

In [ ]:
import torch

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from shorts_intelligence.analyzer import analyze_video
from shorts_intelligence.config import AnalyzerConfig
from shorts_intelligence.discovery.models import DiscoveredCandidate

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ASR_MODEL = "small"
ASR_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_DURATION_SECONDS = 180.0
MAX_FILE_SIZE_MB = 1000
MAX_KEYFRAMES = 24
MAX_OCR_FRAMES = 12

# Stop before the nominal Colab free-session limit to allow checkpointing.
SESSION_BUDGET_SECONDS = 4 * 60 * 60 + 30 * 60
SESSION_STARTED = time.monotonic()

ANALYZER_CONFIG = AnalyzerConfig(
    max_file_size_mb=MAX_FILE_SIZE_MB,
    max_duration_seconds=MAX_DURATION_SECONDS,
    max_keyframes=MAX_KEYFRAMES,
    max_ocr_frames=MAX_OCR_FRAMES,
    asr_model=ASR_MODEL,
    asr_device=ASR_DEVICE,
    asr_timeout_seconds=300,
    enable_asr=True,
    enable_ocr=True,
    enable_audio_analysis=True,
    keep_artifacts=False,
)

print("Analyzer configured | ASR:", ASR_MODEL, "| device:", ASR_DEVICE)


## 5. Optional authentication files

Place authentication files in `MyDrive/ShortsLM_Personal_Pilot/secrets/`:

- `youtube_cookies.txt` — optional Netscape-format cookies exported from your own signed-in YouTube browser. Required only if YouTube challenges anonymous Colab requests.
- `instagram.session` — optional Instaloader session file created by Instaloader for the same username.

You do **not** need either file if the relevant public video downloads anonymously. The notebook will not ask for passwords unless you explicitly choose to refresh an invalid Instagram session.


In [ ]:
YOUTUBE_COOKIE_FILE = SECRETS_DIR / "youtube_cookies.txt"
IG_SESSION_FILE = SECRETS_DIR / "instagram.session"
IG_SESSION_USERNAME = ""

# Optional one-time upload helper. If a file already exists in Drive, it is reused.
if not YOUTUBE_COOKIE_FILE.exists() or not IG_SESSION_FILE.exists():
    print("Optional: upload youtube_cookies.txt and/or instagram.session.")
    print("Click Cancel if you do not want to upload anything now.")
    uploaded_auth = files.upload()
    for filename, data in uploaded_auth.items():
        lowered = filename.casefold()
        if lowered == "youtube_cookies.txt":
            YOUTUBE_COOKIE_FILE.write_bytes(data)
        elif lowered == "instagram.session":
            IG_SESSION_FILE.write_bytes(data)

if IG_SESSION_FILE.exists():
    IG_SESSION_USERNAME = input(
        "Instagram username for instagram.session (without @): "
    ).strip().lstrip("@")

print("YouTube cookies available:", YOUTUBE_COOKIE_FILE.exists())
print("Instagram session file available:", IG_SESSION_FILE.exists())


## 6. Validate Instagram session (only if one was supplied)

In [ ]:
import getpass

INSTALOADER_AVAILABLE = True
try:
    import instaloader
except ImportError:
    INSTALOADER_AVAILABLE = False

def build_instaloader():
    return instaloader.Instaloader(
        quiet=True,
        download_pictures=False,
        download_videos=True,
        download_video_thumbnails=False,
        download_comments=False,
        save_metadata=False,
        post_metadata_txt_pattern="",
    )

IG_SESSION_VALID = False

if IG_SESSION_FILE.exists():
    if not IG_SESSION_USERNAME:
        raise RuntimeError("Instagram session file exists but username was not supplied.")

    loader_check = build_instaloader()
    try:
        loader_check.load_session_from_file(
            IG_SESSION_USERNAME,
            filename=str(IG_SESSION_FILE)
        )
        logged_in_as = loader_check.test_login()
        IG_SESSION_VALID = bool(logged_in_as and logged_in_as.casefold() == IG_SESSION_USERNAME.casefold())
    except Exception as exc:
        print("Could not load Instagram session:", type(exc).__name__, str(exc)[:300])

    if not IG_SESSION_VALID:
        print("The supplied Instaloader session is invalid, expired, or belongs to another username.")
        refresh = input("Refresh it using Instaloader's normal login flow now? (y/N): ").strip().lower()
        if refresh == "y":
            loader_login = build_instaloader()
            try:
                # Instaloader prompts securely for password and handles 2FA when required.
                loader_login.interactive_login(IG_SESSION_USERNAME)
                loader_login.save_session_to_file(filename=str(IG_SESSION_FILE))
                verified = loader_login.test_login()
                IG_SESSION_VALID = bool(verified and verified.casefold() == IG_SESSION_USERNAME.casefold())
            except Exception as exc:
                raise RuntimeError(
                    "Instagram session refresh failed. Check account login/2FA and try again."
                ) from exc

    print("Instagram session valid:", IG_SESSION_VALID)
else:
    print("No Instagram session supplied. Public Reels will be attempted without login.")


## 7. Parse and normalize the URL lists

In [ ]:
YT_RE = re.compile(
    r"(?:youtu\.be/|youtube\.com/(?:shorts/|watch\?v=|embed/|live/))([A-Za-z0-9_-]{11})",
    re.I
)
IG_RE = re.compile(
    r"instagram\.com/(?:reel|reels|p|tv)/([A-Za-z0-9_-]+)",
    re.I
)

def read_source_file(path):
    if path is None:
        return []
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]

def normalize_url(raw_url, platform):
    raw_url = raw_url.strip()
    if platform == "youtube":
        match = YT_RE.search(raw_url)
        if not match:
            raise ValueError(f"Unrecognized YouTube URL: {raw_url}")
        source_id = match.group(1)
        return source_id, f"https://www.youtube.com/shorts/{source_id}"

    match = IG_RE.search(raw_url)
    if not match:
        raise ValueError(f"Unrecognized Instagram URL: {raw_url}")
    source_id = match.group(1)
    return source_id, f"https://www.instagram.com/reel/{source_id}/"

items = []
seen = set()
invalid_urls = []

for platform, path in (("instagram", ig_file), ("youtube", yt_file)):
    for raw in read_source_file(path):
        try:
            source_id, canonical_url = normalize_url(raw, platform)
            key = (platform, source_id)
            if key not in seen:
                seen.add(key)
                items.append({
                    "platform": platform,
                    "source_id": source_id,
                    "url": canonical_url,
                })
        except Exception as exc:
            invalid_urls.append({"url": raw, "platform": platform, "error": str(exc)})

if not items:
    raise RuntimeError("No valid video URLs were found in the supplied TXT files.")

source_urls = [item["url"] for item in items]
ITEM_BY_URL = {item["url"]: item for item in items}

print("Unique valid URLs:", len(items))
print("YouTube:", sum(x["platform"] == "youtube" for x in items))
print("Instagram:", sum(x["platform"] == "instagram" for x in items))
print("Invalid URL lines:", len(invalid_urls))
for bad in invalid_urls[:10]:
    print("INVALID:", bad["url"], "|", bad["error"])


## 8. Initialize resumable state

In [ ]:
def read_lines(path):
    if not path.exists():
        return []
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]

def write_lines_atomic(path, values):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        "".join(str(value) + "\n" for value in values),
        encoding="utf-8"
    )
    temporary.replace(path)

def append_jsonl(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(payload, ensure_ascii=False) + "\n")
        stream.flush()

def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8"
    )
    temporary.replace(path)

completed_set = set(read_lines(COMPLETED_FILE))
# Ignore stale/foreign URLs in an old remaining file; source TXT files are authoritative.
remaining_set = set(read_lines(REMAINING_FILE))
pending_urls = [
    url for url in source_urls
    if url not in completed_set
]
write_lines_atomic(REMAINING_FILE, pending_urls)
write_lines_atomic(COMPLETED_FILE, sorted(completed_set))

print("Completed:", len(completed_set))
print("Remaining:", len(pending_urls))
print("Checkpoint files:", COMPLETED_FILE, REMAINING_FILE)


## 9. Configure the private GitHub destinationThe notebook uses `Arhanpg/ShortsLM-Personal-Features` on branch `main` for feature JSON output. The connected GitHub account currently returned **404 Not Found** for this repository. Create it as a private repository (or correct the name) before running the setup cell.Add a NEW fine-grained token as Google Colab Secret `GH_TOKEN`, scoped to this repository with **Contents: Read and write** permission. Revoke the token previously pasted into the conversation. No credential is embedded in this notebook.

In [ ]:
GITHUB_REPO = "Arhanpg/ShortsLM-Personal-Features"GITHUB_BRANCH = "main"GITHUB_FEATURES_PREFIX = "personal_pilot/features"from google.colab import userdataimport getpassimport requests# Never hardcode a personal access token in a notebook.# Store a NEW fine-grained token in Colab Secrets as GH_TOKEN.try:    GH_TOKEN = userdata.get("GH_TOKEN")except Exception:    GH_TOKEN = Noneif not GH_TOKEN:    GH_TOKEN = getpass.getpass("New GitHub token (hidden): ").strip()if not GH_TOKEN:    raise RuntimeError("Missing GH_TOKEN. Add it to Colab Secrets or enter it at the hidden prompt.")GH_API = "https://api.github.com"GH_HEADERS = {    "Authorization": f"Bearer {GH_TOKEN}",    "Accept": "application/vnd.github+json",    "X-GitHub-Api-Version": "2022-11-28",}repo_response = requests.get(    f"{GH_API}/repos/{GITHUB_REPO}", headers=GH_HEADERS, timeout=30)if repo_response.status_code == 404:    raise RuntimeError(        f"GitHub returned 404 for {GITHUB_REPO}. Create this repository as PRIVATE, "        "or correct GITHUB_REPO. Also ensure GH_TOKEN has access to it."    )if repo_response.status_code == 401:    raise RuntimeError("GitHub rejected the token (401). Replace GH_TOKEN with a valid token.")if repo_response.status_code == 403:    raise RuntimeError(        "GitHub denied access (403). Check fine-grained token repository access "        "and Contents read/write permission."    )repo_response.raise_for_status()repo_info = repo_response.json()if repo_info.get("private") is not True:    raise RuntimeError(        "The feature-output repository must be PRIVATE because feature JSON can "        "contain transcript and OCR text."    )branch_response = requests.get(    f"{GH_API}/repos/{GITHUB_REPO}/branches/{GITHUB_BRANCH}",    headers=GH_HEADERS, timeout=30)if branch_response.status_code == 404:    raise RuntimeError(        f"Branch {GITHUB_BRANCH!r} does not exist in {GITHUB_REPO}. "        "Create it or change GITHUB_BRANCH."    )branch_response.raise_for_status()print("Verified private destination:", GITHUB_REPO, GITHUB_BRANCH)

## 10. GitHub commit helper

In [ ]:
def github_commit_json(path_in_repo, payload, message):
    encoded_path = quote(path_in_repo, safe="/")
    api_url = f"{GH_API}/repos/{GITHUB_REPO}/contents/{encoded_path}"

    raw_json = (
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
    ).encode("utf-8")

    body = {
        "message": message,
        "content": base64.b64encode(raw_json).decode("ascii"),
        "branch": GITHUB_BRANCH,
    }

    existing = requests.get(
        api_url,
        headers=GH_HEADERS,
        params={"ref": GITHUB_BRANCH},
        timeout=30
    )

    if existing.status_code == 200:
        body["sha"] = existing.json()["sha"]
    elif existing.status_code != 404:
        raise RuntimeError(
            f"GitHub file lookup failed: HTTP {existing.status_code}: "
            f"{existing.text[:500]}"
        )

    response = requests.put(
        api_url,
        headers=GH_HEADERS,
        json=body,
        timeout=90
    )

    if response.status_code not in (200, 201):
        raise RuntimeError(
            f"GitHub feature commit failed: HTTP {response.status_code}: "
            f"{response.text[:700]}"
        )

    result = response.json()
    return (
        result.get("commit", {}).get("sha"),
        result.get("content", {}).get("html_url"),
    )


## 11. Platform download adapters — supported authentication only

In [ ]:
from yt_dlp import YoutubeDL
from instaloader import Instaloader, Post

class VideoDownloadError(RuntimeError):
    pass

def download_youtube_video(item, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    output_template = str(output_dir / "%(id)s.%(ext)s")

    ydl_options = {
        "outtmpl": output_template,
        "format": "bv*+ba/b",
        "merge_output_format": "mp4",
        "noplaylist": True,
        "quiet": False,
        "no_warnings": False,
        "retries": 2,
        "fragment_retries": 2,
        "socket_timeout": 30,
        "overwrites": True,
        "js_runtimes": {"node": {"path": shutil.which("node")}} if shutil.which("node") else {},
    }

    # Use only the user's own exported browser cookies, if supplied.
    if YOUTUBE_COOKIE_FILE.exists():
        ydl_options["cookiefile"] = str(YOUTUBE_COOKIE_FILE)

    try:
        with YoutubeDL(ydl_options) as ydl:
            info = ydl.extract_info(item["url"], download=True)
            if not isinstance(info, dict):
                raise VideoDownloadError("yt-dlp returned no video metadata.")
            video_id = info.get("id") or item["source_id"]
    except Exception as exc:
        message = str(exc)
        lowered = message.casefold()
        if any(term in lowered for term in (
            "sign in to confirm", "not a bot", "use --cookies",
            "cookies-from-browser", "authentication required"
        )):
            raise VideoDownloadError(
                "YOUTUBE_AUTH_REQUIRED: YouTube challenged this Colab request. "
                "Export your own signed-in YouTube browser cookies in Netscape format "
                "to secrets/youtube_cookies.txt, then rerun. This notebook does not "
                "bypass YouTube's access controls. Original error: " + message[:900]
            ) from exc
        raise VideoDownloadError(f"yt-dlp download failed: {message[:1200]}") from exc

    candidates = list(output_dir.glob(f"{video_id}*.mp4"))
    if not candidates:
        candidates = list(output_dir.glob("*.mp4"))
    candidates = [path for path in candidates if path.is_file() and path.stat().st_size > 0]

    if not candidates:
        raise VideoDownloadError(
            "yt-dlp returned without a usable MP4. Check the yt-dlp/FFmpeg output above."
        )

    return max(candidates, key=lambda path: path.stat().st_size)


def download_instagram_video(item, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)

    loader = Instaloader(
        quiet=False,
        dirname_pattern=str(output_dir / "{target}"),
        filename_pattern="{shortcode}",
        download_pictures=False,
        download_videos=True,
        download_video_thumbnails=False,
        download_comments=False,
        save_metadata=False,
        post_metadata_txt_pattern="",
    )

    if IG_SESSION_FILE.exists():
        if not IG_SESSION_VALID:
            raise VideoDownloadError(
                "INSTAGRAM_SESSION_INVALID: The saved Instaloader session did not validate. "
                "Refresh the session in the Instagram session validation cell."
            )
        try:
            loader.load_session_from_file(
                IG_SESSION_USERNAME,
                filename=str(IG_SESSION_FILE)
            )
        except Exception as exc:
            raise VideoDownloadError(
                f"INSTAGRAM_SESSION_LOAD_FAILED: {type(exc).__name__}: {exc}"
            ) from exc

    try:
        post = Post.from_shortcode(loader.context, item["source_id"])
        if not post.is_video:
            raise VideoDownloadError("Instagram post is not a video Reel.")
        loader.download_post(post, target=item["source_id"])
    except VideoDownloadError:
        raise
    except Exception as exc:
        message = str(exc)
        lowered = message.casefold()
        if any(term in lowered for term in (
            "login_required", "login required", "checkpoint_required",
            "challenge_required", "please wait a few minutes", "429"
        )):
            raise VideoDownloadError(
                "INSTAGRAM_AUTH_OR_RATE_LIMIT: Instagram refused this request. "
                "Check the session, account access, and rate limits. Do not retry rapidly. "
                f"Original error: {message[:900]}"
            ) from exc
        raise VideoDownloadError(
            f"Instaloader failed for this Reel: {message[:1200]}"
        ) from exc

    videos = [
        path for path in output_dir.rglob("*.mp4")
        if path.is_file() and path.stat().st_size > 0
    ]
    if not videos:
        raise VideoDownloadError(
            "Instaloader completed but produced no MP4. The Reel may be unavailable, "
            "restricted, or not accessible to this account."
        )

    return max(videos, key=lambda path: path.stat().st_size)


def download_video(item, output_dir):
    if item["platform"] == "youtube":
        return download_youtube_video(item, output_dir)
    if item["platform"] == "instagram":
        return download_instagram_video(item, output_dir)
    raise ValueError(f"Unsupported platform: {item['platform']}")


## 12. Process one URL: download → analyze → persist → delete → push

In [ ]:
def safe_id(value):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(value))[:120]

def process_one(item):
    platform = item["platform"]
    source_id = item["source_id"]
    url = item["url"]

    local_json = FEATURES_DIR / platform / f"{safe_id(source_id)}.json"
    github_path = (
        f"{GITHUB_FEATURES_PREFIX}/{platform}/{safe_id(source_id)}.json"
    )

    # If extraction already completed but the previous GitHub upload failed,
    # reuse the Drive JSON and do not download the video again.
    if local_json.exists():
        payload = json.loads(local_json.read_text(encoding="utf-8"))
        print("Reusing saved feature JSON:", local_json)
    else:
        temp_dir = Path(tempfile.mkdtemp(
            prefix=f"shortslm_{platform}_{safe_id(source_id)}_"
        ))
        try:
            media_path = download_video(item, temp_dir)
            media_path = Path(media_path).resolve(strict=True)

            if not media_path.is_file() or media_path.stat().st_size == 0:
                raise RuntimeError("Downloader returned an empty or missing media file.")

            if media_path.stat().st_size > MAX_FILE_SIZE_MB * 1024 * 1024:
                raise RuntimeError(
                    f"Downloaded media exceeds {MAX_FILE_SIZE_MB} MB limit."
                )

            print("Downloaded:", media_path.name, "|", round(media_path.stat().st_size / 1e6, 2), "MB")
            report = analyze_video(
                media_path,
                temp_dir / "analysis.json",
                config=ANALYZER_CONFIG
            )

            payload = {
                "schema_version": "shortslm.personal-pilot.v2",
                "record_type": "short_video_feature_observation",
                "platform": platform,
                "source_id": source_id,
                "canonical_url": url,
                "processed_at": datetime.now(timezone.utc).isoformat(),
                "pipeline": {
                    "repository": REPO_URL,
                    "repository_branch": "main",
                    "asr_model": ASR_MODEL,
                    "asr_device": ASR_DEVICE,
                    "max_duration_seconds": MAX_DURATION_SECONDS,
                    "max_keyframes": MAX_KEYFRAMES,
                    "max_ocr_frames": MAX_OCR_FRAMES,
                    "analyzer_config": "repository AnalyzerConfig",
                },
                "analysis": report.model_dump(mode="json"),
            }

            # Persist features to Drive before deleting the downloaded video.
            atomic_json(local_json, payload)
            print("Feature JSON saved to Drive:", local_json)

        finally:
            # Always remove downloaded video, sidecars, and temporary analysis files.
            shutil.rmtree(temp_dir, ignore_errors=True)
            print("Temporary media directory removed.")

    commit_sha, html_url = github_commit_json(
        github_path,
        payload,
        f"Add ShortsLM personal features for {platform}:{source_id}"
    )

    return {
        "platform": platform,
        "source_id": source_id,
        "url": url,
        "local_json": str(local_json),
        "github_path": github_path,
        "commit_sha": commit_sha,
        "github_url": html_url,
    }


## 13. Run the sequential, resumable pipeline

In [ ]:
successes = 0
failures = 0
attempted_this_session = set()

# Rebuild the queue from the two input files and persistent completion ledger.
completed_set = set(read_lines(COMPLETED_FILE))
queue = [
    url for url in source_urls
    if url not in completed_set
]
write_lines_atomic(REMAINING_FILE, queue)

print("=" * 70)
print("SHORTSLM PERSONAL FEATURE EXTRACTION")
print("=" * 70)
print("Source URLs:", len(source_urls))
print("Already completed:", len(completed_set))
print("To process:", len(queue))
print("One video at a time; no arbitrary URL-count cap.")
print("=" * 70)

try:
    while queue:
        if time.monotonic() - SESSION_STARTED >= SESSION_BUDGET_SECONDS:
            print("Colab safety time budget reached. Remaining URLs checkpointed.")
            break

        url = queue[0]
        item = ITEM_BY_URL[url]
        print(
            f"\\n[{len(completed_set) + 1}/{len(source_urls)}] "
            f"{item['platform']} | {item['source_id']} | remaining: {len(queue)}"
        )

        try:
            result = process_one(item)

            # Only mark completed after GitHub confirms the commit.
            completed_set.add(url)
            append_jsonl(PUSH_LOG, {
                **result,
                "pushed_at": datetime.now(timezone.utc).isoformat(),
            })

            queue.pop(0)
            write_lines_atomic(COMPLETED_FILE, sorted(completed_set))
            write_lines_atomic(REMAINING_FILE, queue)

            successes += 1
            print("PUSHED:", result["github_url"])
            print("COMMIT:", result["commit_sha"])

        except Exception as exc:
            failures += 1
            error_text = f"{type(exc).__name__}: {exc}"

            append_jsonl(FAILED_LOG, {
                "url": url,
                "platform": item["platform"],
                "source_id": item["source_id"],
                "error": error_text,
                "at": datetime.now(timezone.utc).isoformat(),
            })

            print("FAILED:", error_text[:1400])
            print("URL remains in remaining.txt. Pipeline is stopping to avoid repeated requests.")
            write_lines_atomic(REMAINING_FILE, queue)
            break

finally:
    write_lines_atomic(COMPLETED_FILE, sorted(completed_set))
    write_lines_atomic(REMAINING_FILE, queue)

    print("\n" + "=" * 70)
    print("SESSION SUMMARY")
    print("=" * 70)
    print("Successful GitHub pushes:", successes)
    print("Failures:", failures)
    print("Completed total:", len(read_lines(COMPLETED_FILE)))
    print("Remaining total:", len(read_lines(REMAINING_FILE)))
    print("Completed file:", COMPLETED_FILE)
    print("Remaining file:", REMAINING_FILE)
    print("Failure log:", FAILED_LOG)
    print("GitHub push log:", PUSH_LOG)
    print("=" * 70)


## 14. Inspect progress or resume after a Colab reconnect

After reconnecting, mount Drive and rerun setup/configuration cells through the URL parsing and GitHub setup cells. Then rerun the processing cell. The notebook skips URLs listed in `completed.txt` and reuses any feature JSON already saved to Drive if a previous GitHub push failed.

Persistent files:
- `state/completed.txt`
- `state/remaining.txt`
- `state/failed.jsonl`
- `state/github_push_log.jsonl`
- `features/youtube/<video_id>.json`
- `features/instagram/<shortcode>.json`

**Do not upload the `secrets/` directory to GitHub.**


In [ ]:
print("COMPLETED:")
print("\n".join(read_lines(COMPLETED_FILE)) or "(none)")
print("\nREMAINING:")
print("\n".join(read_lines(REMAINING_FILE)) or "(none)")
print("\nFeature JSON files:", sum(1 for _ in FEATURES_DIR.rglob("*.json")))
